## Using TorchIO to replace the standard pytorch dataloader and the custom patch-processor. 

In [1]:
import torch
import torchio as tio
import os
import numpy as np
from skimage import io

In [2]:
data_source_folder = '/scratch/park/data/Kaarjel/1st_training/confocal'
data_target_folder = '/scratch/park/data/Kaarjel/1st_training/STED'

In [3]:
import os

# Fetch all 'tif' images in the data_source_folder
tif_images_source = [os.path.join(data_source_folder, f) for f in os.listdir(data_source_folder) if f.endswith('.tif')]
print(tif_images_source)
print(f"image # from source: {len(tif_images_source)}")

tif_images_target = [os.path.join(data_target_folder, f) for f in os.listdir(data_target_folder) if f.endswith('.tif')]
print(tif_images_target)
print(f"image # from target: {len(tif_images_target)}")

In [4]:
test_source_img_raw = io.imread(tif_images_source[0])
test_target_img_raw= io.imread(tif_images_target[0])

test_source_img = np.transpose(test_source_img_raw, axes=(2, 1, 0))
test_target_img = np.transpose(test_target_img_raw, axes=(2, 1, 0))

# Add an extra dimension to the tensors
test_source_img = torch.unsqueeze(torch.tensor(test_source_img), 0)
test_target_img = torch.unsqueeze(torch.tensor(test_target_img), 0)
# print(test_source_img.shape)

### Note: TorchIO handles the dimension as in reverse: (C, W, H, D) or (C, X, Y, Z). 

In [28]:
subject_source = tio.Subject(
    image=tio.ScalarImage(tensor=test_source_img),
)
patch_size = 120, 120, 25
patch_overlap = 20

In [29]:
grid_sampler = tio.inference.GridSampler(subject_source, patch_size, patch_overlap)
patch_loader = tio.SubjectsLoader(grid_sampler, batch_size=1)
aggregator = tio.inference.GridAggregator(grid_sampler)

In [30]:
len(patch_loader)

In [31]:
grid_sampler[0].image.plot()

In [32]:

# Iterate over the patches and place them back into the reconstructed volume
with torch.no_grad():
    for patch in patch_loader:
        patch_data = patch['image'][tio.DATA] # (B, C, H, W, D)
        locations = patch[tio.LOCATION] # (B, 6)
        # print(patch_data.shape)
        aggregator.add_batch(patch_data, locations)

In [33]:
output_tensor = aggregator.get_output_tensor()

In [34]:
output_tensor.shape

In [35]:
subject_source.add_image(tio.ScalarImage(tensor=test_source_img), 'reconstructed_image')

In [37]:
subject_source['image'].plot()
subject_source['reconstructed_image'].plot()